In [373]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [374]:
import pandas as pd
temp_df = pd.read_csv('../data/SPY_data.csv', nrows=0)
print(temp_df.columns.tolist())


['Price', 'Close', 'High', 'Low', 'Open', 'Volume']


In [375]:
df = pd.read_csv("../data/SPY_data.csv", index_col="Price", parse_dates=True)
print(df.shape)
print(df.head())

(2518, 5)
                         Close                High                 Low  \
Price                                                                    
Ticker                     SPY                 SPY                 SPY   
Date                       NaN                 NaN                 NaN   
2015-01-02  170.12498474121094  171.32579919572484  169.08980858140774   
2015-01-05  167.05258178710938  169.24715009688967   166.7461737006491   
2015-01-06  165.47911071777344  167.88071412827958  164.68408994172034   

                          Open     Volume  
Price                                      
Ticker                     SPY        SPY  
Date                       NaN        NaN  
2015-01-02  170.91172873180355  121465900  
2015-01-05   169.0815244457335  169632600  
2015-01-06  167.35898134910641  209151400  


C:\Users\Welcome\AppData\Local\Temp\ipykernel_17512\2374231573.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv("../data/SPY_data.csv", index_col="Price", parse_dates=True)


In [376]:
print(df.iloc[2].apply(type))   #numerical numbers are read as strings

Close     <class 'str'>
High      <class 'str'>
Low       <class 'str'>
Open      <class 'str'>
Volume    <class 'str'>
Name: 2015-01-02, dtype: object


In [377]:
# Convert all columns to numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

adding a new column as "return" and pct_change() calculates how much the price moved in percentage terms each day. If the stock went from $100 to $102, the return is 0.02 (2%). This is the core signal everything else is built from.

In [378]:
df["return"] = df["Close"].pct_change()

In [379]:
print(df.head())

                 Close        High         Low        Open       Volume  \
Price                                                                     
Ticker             NaN         NaN         NaN         NaN          NaN   
Date               NaN         NaN         NaN         NaN          NaN   
2015-01-02  170.124985  171.325799  169.089809  170.911729  121465900.0   
2015-01-05  167.052582  169.247150  166.746174  169.081524  169632600.0   
2015-01-06  165.479111  167.880714  164.684090  167.358981  209151400.0   

              return  
Price                 
Ticker           NaN  
Date             NaN  
2015-01-02       NaN  
2015-01-05 -0.018060  
2015-01-06 -0.009419  


Financial markets often display short-term patterns that lag variables expose: <br>
**Momentum:** If lag1 and lag2 are highly positive, the asset might be in a strong upward trend.<br>
**Mean Reversion:** If an asset went up too high over lag1, lag2, and lag3, it might be due for a downward correction.<br><br>
shift(1) moves the column down by one row — so today's row now contains yesterday's value. You're giving the model a short memory of recent price movements.

In [380]:
df["lag1"] = df["return"].shift(1)  # yesterday's return
df["lag2"] = df["return"].shift(2)  # two days ago
df["lag3"] = df["return"].shift(3)  # three days ago
df["lag5"] = df["return"].shift(5)  # five days ago
df["lag10"] = df["return"].shift(10)  # ten days ago

In [381]:
df["sma10"]  = df["Close"].rolling(10).mean() # Average of the last 10 days
df["sma20"] = df["Close"].rolling(20).mean() # Average of the last 20 days
df["sma50"] = df["Close"].rolling(50).mean() # Average of the last 50 days

#### Price relative to moving average
| Result | What it means |
|---|---|
| **= 1.0** | Stock is trading *exactly* at its average — perfectly normal |
| **> 1.0** (e.g. 1.15) | Stock is trading *above* its average — it's been running hot |
| **< 1.0** (e.g. 0.85) | Stock is trading *below* its average — it's been cooling off |

This code is comparing today's stock price to the moving averages you calculated before. It's asking:

In [382]:

df["price_to_sma10"] = df["Close"] / df["sma10"]
df["price_to_sma20"] = df["Close"] / df["sma20"]
df["sma10_sma20_r"] = df["sma10"] / df["sma20"]

Standard deviation of returns over the last 10 or 20 days. High volatility means the stock has been jumping around a lot lately. Low volatility means it's been calm. The model can use this to adjust its confidence.

In [383]:
df["volatility_10"] = df["return"].rolling(10).std()
df["volatility_20"] = df["return"].rolling(20).std()
df["vol_ratio"]     = df["volatility_10"] / df["volatility_20"]

RSI(Relative Strength Index) runs from 0 to 100. Above 70 generally means the stock has been rising fast and might be due for a pullback. Below 30 means the opposite. It's one of the most commonly used momentum indicators.

In [384]:
delta = df["Close"].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
df["rsi_14"] = 100 - (100 / (1 + gain / loss))

In [385]:
# df["volume_change"] = df["Volume"].pct_change()
df["volume_ma20"]   = df["Volume"].rolling(20).mean()
df["volume_ratio"]  = df["Volume"] / df["Volume"].rolling(20).mean()
df["vol_confirmed_move"] = df["lag1"] * df["volume_ratio"]

# volume ratio 2.0 --> 2x the normal volume (something big is happening)
# volume ratio 0.5 --> half the normal volume (quiet day)

In [386]:
df["target"] = (df["return"].shift(-1) > 0).astype(int)

# shift(-1) moves the column up by one row — so today's row gets tomorrow's return value
# > 0 turns it into True/False — did it go up?
# .astype(int) converts True → 1, False → 0

In [387]:
df.dropna(inplace=True)
print(f"Dataset shape after dropping NaNs: {df.shape}")
print(df.head())

Dataset shape after dropping NaNs: (2467, 25)
                 Close        High         Low        Open       Volume  \
Price                                                                     
2015-03-16  172.733612  172.824708  170.481068  171.184993  136099200.0   
2015-03-17  172.220245  172.601183  171.408658  171.996644   94510400.0   
2015-03-18  174.290558  174.961350  171.110487  171.748159  228808500.0   
2015-03-19  173.495560  174.298858  173.106332  173.876510  117917300.0   
2015-03-20  175.027008  175.534430  174.261719  174.444724  177715100.0   

              return      lag1      lag2      lag3      lag5  ...  \
Price                                                         ...   
2015-03-16  0.013360 -0.006132  0.012714 -0.002342  0.004145  ...   
2015-03-17 -0.002972  0.013360 -0.006132  0.012714 -0.016222  ...   
2015-03-18  0.012021 -0.002972  0.013360 -0.006132 -0.002342  ...   
2015-03-19 -0.004561  0.012021 -0.002972  0.013360  0.012714  ...   
2015-03-20  0.

Lags → What has the stock been doing recently? <br>
SMA / price ratio → Is it trending up or down overall? <br>
RSI → Has it moved too far too fast? <br>
Volatility → How risky is the current environment? <br>
Volume → Is there real conviction behind recent moves? <br>

In [388]:
print("Features created:")
print(df.columns.tolist())
print(f"\nDate range: {df.index[0]} to {df.index[-1]}")
print(f"Total rows: {len(df)}")

Features created:
['Close', 'High', 'Low', 'Open', 'Volume', 'return', 'lag1', 'lag2', 'lag3', 'lag5', 'lag10', 'sma10', 'sma20', 'sma50', 'price_to_sma10', 'price_to_sma20', 'sma10_sma20_r', 'volatility_10', 'volatility_20', 'vol_ratio', 'rsi_14', 'volume_ma20', 'volume_ratio', 'vol_confirmed_move', 'target']

Date range: 2015-03-16 to 2024-12-31
Total rows: 2467


In [389]:
df.to_csv("../data/SPY_features.csv")
print("Saved to ../data/SPY_features.csv")

Saved to ../data/SPY_features.csv
